### UNESCO DATA

In [2]:
import os

import pandas as pd
import requests

## repeated printouts
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [6]:
# Used UNESCO APIs to pull all indicators
i_q = (
'https://api.uis.unesco.org/'
'api/public/definitions/indicators')
res3 = requests.get(i_q, params = {"disaggregations": "false"})
res3.raise_for_status()
indicators_df = pd.DataFrame(res3.json())[["indicatorCode", "name"]]
indicators_df.shape
indicators_df.head()

(5556, 2)

,indicatorCode,name
0,10,Official entrance age to early childhood educa...
1,10403,Start month of the academic school year (terti...
2,10404,End month of the academic school year (tertiar...
3,10405,Start of the academic school year (tertiary ed...
4,10406,End of the academic school year (tertiary educ...


In [7]:
#unesco api to pull geounits(countries) and their codes, which will help in merging with the World Bank data as to what constitutes a developing country
c_q = (
'https://api.uis.unesco.org/'
'api/public/definitions/geounits')
res2 = requests.get(c_q)
res2.raise_for_status()
resp2_j = res2.json()
resp2_j_df = pd.DataFrame(resp2_j)
resp2_j_df.type.value_counts()

## countries dataframe
uis_countries = (resp2_j_df[resp2_j_df.type == "NATIONAL"]
                 [["id", "name"]]
                 .rename(columns = {"id": "iso3", "name": "country"}))
uis_countries.shape
uis_countries.head()

type
NATIONAL    241
REGIONAL    226
Name: count, dtype: int64

(241, 2)

,iso3,country
0,ABW,Aruba
1,AFG,Afghanistan
2,AGO,Angola
3,AIA,Anguilla
4,ALA,Åland Islands


In [9]:
#Identified indicator variables that will help us in answering question on education literacy that we will then pull out using the fuction below 
uis_vars = {
    'CR.1': 'compl_primary',                     
    'CR.2': 'compl_lowersec',
    'CR.3': 'compl_uppersec',
    'CR.MOD.1': 'compl_primary_mod',             
    'CR.MOD.2': 'compl_lowersec_mod',
    'CR.MOD.3': 'compl_uppersec_mod',
    'ROFST.1.CP': 'oos_rate_primary',
    'ROFST.2.CP': 'oos_rate_lowersec',
    'ROFST.3.CP': 'oos_rate_uppersec',
    'NERT.1.CP': 'net_enrol_primary',
    'NERT.2.CP': 'net_enrol_lowersec',
    'NERT.3.CP': 'net_enrol_uppersec',
    'READ.PRIMARY': 'min_prof_read_primary',
    'MATH.PRIMARY': 'min_prof_math_primary',
    'READ.LOWERSEC': 'min_prof_read_lowersec',
    'MATH.LOWERSEC': 'min_prof_math_lowersec',
    'SCHBSP.1.WINTERN': 'school_internet_primary',
    'SCHBSP.2.WINTERN': 'school_internet_lowersec',
    'SCHBSP.3.WINTERN': 'school_internet_uppersec',
    'SCHBSP.2.WCOMPUT': 'school_computers_lowersec',
    'SCHBSP.2.WELEC': 'school_electricity_lowersec',
    'XGDP.FSGOV': 'edu_spend_pct_gdp',
    'TRTP.2': 'qualified_teachers_lowersec',
    'CR.2.Q1': 'compl_lowersec_poorest',
    'CR.2.Q5': 'compl_lowersec_richest',
    'CR.2.F': 'compl_lowersec_f',
    'CR.2.M': 'compl_lowersec_m',
    'CR.3.F': 'compl_uppersec_f',
    'CR.3.M': 'compl_uppersec_m',
    'CR.MOD.3.F': 'compl_uppersec_mod_f',
    'CR.MOD.3.M': 'compl_uppersec_mod_m',
    'CR.3.GPIA': 'compl_uppersec_gpia',
    'ROFST.3.F.CP': 'oos_rate_uppersec_f',
    'ROFST.3.M.CP': 'oos_rate_uppersec_m',
    'GER.5T8.F': 'tertiary_enrol_f',             
    'GER.5T8.M': 'tertiary_enrol_m',
    'LR.AG15T24.F': 'youth_literacy_f',          
    'LR.AG15T24.M': 'youth_literacy_m',
    'EA.3T8.AG25T99.F': 'attain_uppersec_25plus_f',
    'EA.3T8.AG25T99.M': 'attain_uppersec_25plus_m',
    'LR.AG15T99.F': 'adult_literacy_f'}

In [11]:

# Function to pull indicators, we will use  considering years from 2000 to 2024, but will depend on the merged data we have from other sources 
def pull_indicator(indicator_code, start = 2000, end = 2024):

    url = (
    'https://api.uis.unesco.org/'
    'api/public/data/indicators')

    params = {
        "indicator": indicator_code,
        "geoUnitType": "NATIONAL",    ## countries only, no regional averages
        "start": start,
        "end": end
    }

    resp = requests.get(url, params = params)
    resp.raise_for_status()

    resp_j = resp.json()

    if len(resp_j["records"]) == 0:
        print("No data for", indicator_code, resp_j["hints"])
        return pd.DataFrame()

    return pd.DataFrame(resp_j["records"])


    

In [12]:
uis_raw = pd.concat([pull_indicator(c) for c in uis_vars.keys()], ignore_index = True)
uis_raw.shape
uis_raw.head()

(85626, 6)

,indicatorId,geoUnit,year,value,magnitude,qualifier
0,CR.1,AFG,2011,40.725689,NaN,None
1,CR.1,AFG,2022,44.242260,NaN,None
2,CR.1,AGO,2015,59.955662,NaN,None
3,CR.1,AGO,2023,55.250381,NaN,None
4,CR.1,ALB,2000,95.190000,NaN,None


In [13]:
## tidy long format with readable names
uis_long = pd.DataFrame({"iso3": uis_raw.geoUnit,
                         "year": uis_raw.year,
                         "indicator_code": uis_raw.indicatorId,
                         "variable": uis_raw.indicatorId.map(uis_vars),
                         "value": uis_raw.value,
                         "qualifier": uis_raw.qualifier})    ## UNESCO flags, e.g. estimates

## add country names from UNESCO's list
uis_long = uis_long.merge(uis_countries, on = "iso3", how = "left")
uis_long.head()

,iso3,year,indicator_code,variable,value,qualifier,country
0,AFG,2011,CR.1,compl_primary,40.725689,None,Afghanistan
1,AFG,2022,CR.1,compl_primary,44.242260,None,Afghanistan
2,AGO,2015,CR.1,compl_primary,59.955662,None,Angola
3,AGO,2023,CR.1,compl_primary,55.250381,None,Angola
4,ALB,2000,CR.1,compl_primary,95.190000,None,Albania


## Subset Data to developing countries using World Bank Classification

In [15]:
wb_q = (
'https://api.worldbank.org/'
'v2/country')
wb_resp = requests.get(wb_q, params = {"format": "json", "per_page": 400})
wb_resp.raise_for_status()

## the World Bank returns [page info, list of countries]
wb_j = wb_resp.json()
wb_j[1][1]

{'id': 'AFE',
 'iso2Code': 'ZH',
 'name': 'Africa Eastern and Southern',
 'region': {'id': 'NA', 'iso2code': 'NA', 'value': 'Aggregates'},
 'adminregion': {'id': '', 'iso2code': '', 'value': ''},
 'incomeLevel': {'id': 'NA', 'iso2code': 'NA', 'value': 'Aggregates'},
 'lendingType': {'id': '', 'iso2code': '', 'value': 'Aggregates'},
 'capitalCity': '',
 'longitude': '',
 'latitude': ''}

In [17]:
## keep code, region, income group; region id 'NA' = aggregates like "World"
wb_groups = pd.DataFrame([{"iso3": c["id"],
                           "wb_region": c["region"]["value"].strip(),
                           "income_group": c["incomeLevel"]["value"]}
                          for c in wb_j[1] if c["region"]["id"] != "NA"])

developing_groups = ["Low income", "Lower middle income", "Upper middle income"]
wb_groups["developing"] = wb_groups.income_group.isin(developing_groups)

## the World Bank recently moved Afghanistan and Pakistan out of "South Asia",
## so we define South Asia ourselves as the 8 SAARC countries (needed for Q1c / Q2c)
wb_groups["south_asia"] = wb_groups.iso3.isin(
    ["AFG", "BGD", "BTN", "IND", "LKA", "MDV", "NPL", "PAK"])

wb_groups.income_group.value_counts()

income_group
High income            86
Upper middle income    59
Lower middle income    47
Low income             25
Name: count, dtype: int64

In [18]:
## merge and check
uis_merged = uis_long.merge(wb_groups, on = "iso3", how = "left", indicator = "merge_status")
uis_merged.merge_status.value_counts()

## UNESCO countries with no World Bank group (small territories) - dropped
uis_merged[uis_merged.merge_status == "left_only"].country.unique()

merge_status
both          84560
left_only      1066
right_only        0
Name: count, dtype: int64

<ArrowStringArray>
[             'Cook Islands',                'Montserrat',
                   'Tokelau', 'Wallis and Futuna Islands',
                      'Niue',                  'Anguilla',
                  'Holy See',     'Sudan (pre-secession)']
Length: 8, dtype: str

In [21]:
## keep developing countries only
dev_long = (uis_merged[uis_merged.developing == True]
            [["iso3", "country", "wb_region", "south_asia", "income_group",
              "year", "indicator_code", "variable", "value", "qualifier"]]
            .sort_values(["iso3", "year", "variable"])
            .reset_index(drop = True))

dev_long.shape
dev_long.iso3.nunique()
dev_long.groupby("income_group").iso3.nunique()
dev_long[dev_long.south_asia].country.unique()

#one row per country year
dev_wide = (dev_long
            .pivot_table(index = ["iso3", "country", "wb_region", "south_asia",
                                  "income_group", "year"],
                         columns = "variable", values = "value")
            .reset_index())

## put indicator columns in the same order as the dictionary
id_cols = ["iso3", "country", "wb_region", "south_asia", "income_group", "year"]
dev_wide = dev_wide[id_cols + [v for v in uis_vars.values() if v in dev_wide.columns]]

dev_wide.shape
dev_wide.head()

(52006, 10)

130

income_group
Low income             25
Lower middle income    47
Upper middle income    58
Name: iso3, dtype: int64

<ArrowStringArray>
['Afghanistan',  'Bangladesh',      'Bhutan',       'India',   'Sri Lanka',
    'Maldives',       'Nepal',    'Pakistan']
Length: 8, dtype: str

(3179, 47)

variable,iso3,country,wb_region,south_asia,income_group,year,compl_primary,compl_lowersec,compl_uppersec,compl_primary_mod,...,compl_uppersec_gpia,oos_rate_uppersec_f,oos_rate_uppersec_m,tertiary_enrol_f,tertiary_enrol_m,youth_literacy_f,youth_literacy_m,attain_uppersec_25plus_f,attain_uppersec_25plus_m,adult_literacy_f
0,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",True,Low income,2000,NaN,NaN,NaN,18.590000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",True,Low income,2001,NaN,NaN,NaN,19.480000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",True,Low income,2002,NaN,NaN,NaN,20.709999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",True,Low income,2003,NaN,NaN,NaN,22.190001,...,NaN,NaN,NaN,0.57137,2.14712,NaN,NaN,NaN,NaN,NaN
4,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",True,Low income,2004,NaN,NaN,NaN,24.049999,...,NaN,NaN,NaN,0.57816,2.17250,NaN,NaN,NaN,NaN,NaN


In [22]:
## Codebook 
names = indicators_df.set_index("indicatorCode")["name"]

codebook = (dev_long.groupby(["indicator_code", "variable"])
            .agg(countries = ("iso3", "nunique"),
                 country_years = ("value", "count"),
                 first_year = ("year", "min"),
                 last_year = ("year", "max"))
            .reset_index())
codebook["description"] = codebook.indicator_code.map(names)

## keep dictionary order
codebook["order"] = codebook.variable.map({v: i for i, v in enumerate(uis_vars.values())})
codebook = codebook.sort_values("order").drop(columns = "order").reset_index(drop = True)

pd.options.display.max_colwidth = 80
codebook

,indicator_code,variable,countries,country_years,first_year,last_year,description
0,CR.1,compl_primary,123,994,2000,2024,"Completion rate, primary education, both sexes (%)"
1,CR.2,compl_lowersec,123,1012,2000,2024,"Completion rate, lower secondary education, both sexes (%)"
2,CR.3,compl_uppersec,125,993,2000,2024,"Completion rate, upper secondary education, both sexes (%)"
3,CR.MOD.1,compl_primary_mod,117,2925,2000,2024,"Completion rate, primary education, both sexes (modelled data) (%)"
4,CR.MOD.2,compl_lowersec_mod,114,2850,2000,2024,"Completion rate, lower secondary education, both sexes (modelled data) (%)"
5,CR.MOD.3,compl_uppersec_mod,114,2850,2000,2024,"Completion rate, upper secondary education, both sexes (modelled data) (%)"
6,ROFST.1.CP,oos_rate_primary,123,2046,2000,2024,"Out-of-school rate for children of primary school age, both sexes (%)"
7,ROFST.2.CP,oos_rate_lowersec,120,1594,2000,2024,"Out-of-school rate for adolescents of lower secondary school age, both sexes..."
8,ROFST.3.CP,oos_rate_uppersec,121,1605,2000,2024,"Out-of-school rate for youth of upper secondary school age, both sexes (%)"
9,NERT.1.CP,net_enrol_primary,124,2118,2000,2024,"Total net enrolment rate, primary, both sexes (%)"


In [25]:
#saving data file and codebook
os.makedirs("data", exist_ok = True)
dev_wide.to_csv("data/unesco_developing_countries.csv", index = False)
codebook.to_csv("data/unesco_codebook.csv", index = False)